In [ ]:
pip install --quiet "numpy<2" opencv-python torch torchvision captum umap-learn

In [11]:
import sys
sys.path.insert(1, '../')
import helpers
from helpers.crossattention import CrossAttentionSiamese

import os
import torch
from torchvision import transforms
from pathlib import Path

# Import analysis functions (save the artifact as retroactive_analysis.py)
import retroactive_analysis as ra

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [3]:
models_path = "logs/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/"
CHECKPOINT_DIR = os.path.join(models_path, "checkpoints")
train_set = models_path.split("/")[2]
MODEL_TYPE = models_path.split("/")[3]
image_process = models_path.split("/")[4]
run_name = models_path.split("/")[5]
MODEL_NAME = run_name.split("_")[0]
if MODEL_TYPE == "cross_attention":
    weights = run_name.split("_")[0]
    NUM_ATTN_LAYERS = int(run_name.split("_")[2].split("L")[0][-1:])
    NUM_HEADS = int(run_name.split("_")[2].split("L")[1][0])
    use_symm = True if run_name.split("_")[4] == "sym" else False
    run_id = models_path.strip("/")[-1:]
else:
    raise NotImplementedError(f"Model type \"{MODEL_NAME}\" unsupported! Model Type must be: cross_attention ")
RESIZE_DIM = 224
OUTPUT_DIR = os.path.join(models_path.replace("logs","plots"), "analysis_outputs")
FREEZE_BACKBONE = True
BATCH_SIZE = 64
WORKERS = 1
DATASET_BASE_PATH = f"../split_node21_sets/{image_process}"

In [4]:
ANALYZE_BEST_ONLY = False

In [5]:
if MODEL_NAME == "single":
    transform = transforms.Compose([
        transforms.Grayscale(1),
        transforms.Resize((RESIZE_DIM, RESIZE_DIM)),
        transforms.ToTensor(),
    ])
else:
    transform = transforms.Compose([
        transforms.Resize((RESIZE_DIM, RESIZE_DIM)),
        transforms.ToTensor(),
    ])


In [6]:
datasets_config = {
    'train-chestxray14': f"{DATASET_BASE_PATH}/chestxray14/train",
    'test-chestxray14': f"{DATASET_BASE_PATH}/chestxray14/test",
    'test-jsrt': f"{DATASET_BASE_PATH}/jsrt/test",
    'test-padchest': f"{DATASET_BASE_PATH}/padchest/test",
}

In [7]:
dataloaders = {}
for name, path in datasets_config.items():
    print(f"\nLoading {name}...")
    dataloaders[name] = helpers.dataloading.load_image_pair_dataset(
        dataset_path=path,
        batch_size=BATCH_SIZE,
        crop_size=RESIZE_DIM,
        symmetrical_transforms=False,
        class_to_idx={'nodule': 1, 'normal': 0},
        transform=transform,
        cache_in_ram=False,
        single=(MODEL_NAME == "single"),
        num_workers=WORKERS
    )


Loading train-chestxray14...


Loading dataset: 100%|██████████| 2/2 [00:05<00:00,  2.88s/it]


Total pairs: 1239
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 420
  normal (idx=0): 819

Loading test-chestxray14...


Loading dataset: 100%|██████████| 2/2 [00:02<00:00,  1.15s/it]


Total pairs: 531
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 180
  normal (idx=0): 351

Loading test-jsrt...


Loading dataset: 100%|██████████| 2/2 [00:00<00:00,  9.27it/s]


Total pairs: 72
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 44
  normal (idx=0): 28

Loading test-padchest...


Loading dataset: 100%|██████████| 2/2 [00:05<00:00,  2.69s/it]

Total pairs: 504
Number of classes: 2
Class mapping: {'nodule': 1, 'normal': 0}

Pairs per class:
  nodule (idx=1): 94
  normal (idx=0): 410


In [8]:
print("Building model...")
backbone = helpers.models.load_truncated_model(MODEL_NAME)

if MODEL_TYPE == "siamese":
    model = helpers.models.SiameseNetwork(
        backbone,
        embedding_dim=128,
        freeze_backbone=FREEZE_BACKBONE
    ).to(device)
elif MODEL_TYPE == "cross_attention":
    model = CrossAttentionSiamese(
        backbone,
        embedding_dim=128,
        num_attn_layers=NUM_ATTN_LAYERS,
        num_heads=NUM_HEADS,
        freeze_backbone=FREEZE_BACKBONE
    ).to(device)

print(f"Model: {MODEL_TYPE}")

Building model...


/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/opt/conda/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Model: cross_attention


In [9]:
checkpoint_files = sorted(Path(CHECKPOINT_DIR).glob("checkpoint_epoch_*.pth"))
print(f"\nFound {len(checkpoint_files)} checkpoint(s) to analyze")


Found 30 checkpoint(s) to analyze


In [10]:
for ckpt_path in checkpoint_files:
    results = ra.analyze_single_checkpoint(
    checkpoint_path=str(ckpt_path),
    model=model,
    dataloaders_dict=dataloaders,
    device=device,
    output_dir=OUTPUT_DIR,
    show_plots=False  # Set to False if you don't want plots in notebook
    )
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    for dataset_name, metrics in results.items():
        print(f"\n{dataset_name}:")
        print(f"  Effective Rank: {metrics['effective_rank']:.2f}")
        print(f"  Distance Std: {metrics['distance_std']:.4f}")
        if metrics['normalized_entropy'] is not None:
            print(f"  Attention Entropy: {metrics['normalized_entropy']:.3f}")
    


Analyzing: logs/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/checkpoints/checkpoint_epoch_0.pth
Epoch: 0

--- train-chestxray14 ---


/home/local/data/sophie/contrastive_cxr/notebooks/retroactive_analysis.py:151: RuntimeWarning: invalid value encountered in scalar divide
  normalized_entropy = avg_entropy / max_entropy
/opt/conda/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved: plots/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/analysis_outputs/train-chestxray14_projections_epoch_0.png
Saved: plots/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/analysis_outputs/train-chestxray14_attention_epoch_0.png
  Effective Rank: 3.03
  Distance Std: 0.0303
  Norm Std: 0.0000
  Attention Entropy: nan

--- test-chestxray14 ---


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7f0e4e7aade0>
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1479, in __del__
    self._shutdown_workers()
  File "/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1462, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AssertionError: can only test a child process
/home/local/data/sophie/contrastive_cxr/notebooks/retroactive_analysis.py:151: RuntimeWarning: invalid value encountered in scalar divide
  normalized_entropy = avg_entropy / max_entropy
/opt/conda/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for paralle

Saved: plots/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/analysis_outputs/test-chestxray14_projections_epoch_0.png
Saved: plots/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/analysis_outputs/test-chestxray14_attention_epoch_0.png
  Effective Rank: 2.98
  Distance Std: 0.0293
  Norm Std: 0.0000
  Attention Entropy: nan

--- test-jsrt ---


...

Saved: plots/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/analysis_outputs/test-chestxray14_projections_epoch_90.png
Saved: plots/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/analysis_outputs/test-chestxray14_attention_epoch_90.png
  Effective Rank: 5.66
  Distance Std: 0.5595
  Norm Std: 0.0000
  Attention Entropy: nan

--- test-jsrt ---


/home/local/data/sophie/contrastive_cxr/notebooks/retroactive_analysis.py:151: RuntimeWarning: invalid value encountered in scalar divide
  normalized_entropy = avg_entropy / max_entropy
/opt/conda/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved: plots/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/analysis_outputs/test-jsrt_projections_epoch_90.png
Saved: plots/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/analysis_outputs/test-jsrt_attention_epoch_90.png
  Effective Rank: 3.81
  Distance Std: 0.4400
  Norm Std: 0.0000
  Attention Entropy: nan

--- test-padchest ---


/home/local/data/sophie/contrastive_cxr/notebooks/retroactive_analysis.py:151: RuntimeWarning: invalid value encountered in scalar divide
  normalized_entropy = avg_entropy / max_entropy
/opt/conda/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Saved: plots/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/analysis_outputs/test-padchest_projections_epoch_90.png
Saved: plots/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/analysis_outputs/test-padchest_attention_epoch_90.png
  Effective Rank: 4.08
  Distance Std: 0.4235
  Norm Std: 0.0000
  Attention Entropy: nan

SUMMARY

train-chestxray14:
  Effective Rank: 5.55
  Distance Std: 0.6424
  Attention Entropy: nan

test-chestxray14:
  Effective Rank: 5.66
  Distance Std: 0.5595
  Attention Entropy: nan

test-jsrt:
  Effective Rank: 3.81
  Distance Std: 0.4400
  Attention Entropy: nan

test-padchest:
  Effective Rank: 4.08
  Distance Std: 0.4235
  Attention Entropy: nan


In [ ]:
df = ra.analyze_multiple_checkpoints(
    checkpoint_dir=CHECKPOINT_DIR,
    model=model,
    dataloaders_dict=dataloaders,
    device=device,
    output_dir=OUTPUT_DIR,
    analyze_all=True,  # Analyze all checkpoints
    show_plots=False   # Don't show plots for each checkpoint
)

In [25]:
CHECKPOINT_DIR

'logs/subsets/chestxray14/cross_attention/arch_seg/rad_unfrz_crossattn1L8H_cosine_sym_bsz128_1/checkpoints'

In [ ]:
checkpoint_path = Path(CHECKPOINT_DIR) / "best_model.pth"

results = ra.analyze_single_checkpoint(
    checkpoint_path=str(checkpoint_path),
    model=model,
    dataloaders_dict=dataloaders,
    device=device,
    output_dir=OUTPUT_DIR,
    show_plots=True  # Set to False if you don't want plots in notebook
)